# Actividad 03: Data Marts y Data Warehouse para Parasol Corp

Cada área de Parasol Corp quiere consultar **solo lo suyo**: Experimentos necesita
seguimiento de experimentos, Bioresearch necesita muestras y niveles de riesgo, y
Finanzas necesita presupuestos y gastos. Si todo vive junto, cada área mira tablas
que no le interesan.

**Tu trabajo:** crear 3 **Data Marts** departamentales, cada uno en su propio
**esquema** de la base de datos, con 3 tablas relacionadas. Después construirás el
**Data Warehouse** con **vistas** que integran los datos de los 3 marts.

Caso de negocio completo: [brief de Wesker](../BRIEF.md).


In [ ]:
import duckdb

con = duckdb.connect(':memory:')
print('Base de datos DuckDB en memoria creada.')

## Qué aprenderás

1. Crear **esquemas** (`CREATE SCHEMA`) como namespaces para isolar cada data mart
2. Modelar **3 tablas por mart** con sus PK y FK internas
3. Construir un **Data Warehouse** con **vistas** que integran los marts sin copiar datos


---

## Los esquemas: cada mart en su espacio de nombres

Un **esquema** es un namespace dentro de la base de datos: agrupa tablas y evita
colisiones de nombres. Así `dm_experimentos.experimentos` y `dm_bioresearch.muestras`
conviven sin pisarse, y cada área sabe qué tablas son suyas.

```mermaid
flowchart TB
    E["dm_experimentos<br/>experimentos"]
    M["dm_bioresearch<br/>muestras"]
    G["dm_finance<br/>gastos y proyectos"]
    V1["dw.v_experimentos_full<br/>JOIN por experiment_id y project_id"]
    V2["dw.v_coste_por_laboratorio"]
    V3["dw.v_muestras_alto_riesgo"]
    E --> V1
    M --> V1
    G --> V1
    V1 --> V2
    V1 --> V3
```

Se crea con `CREATE SCHEMA dm_experimentos;` y las tablas se referencian como
`esquema.tabla`.


---

## Parte 1: el data mart `dm_experimentos`

El área de **Experimentos** quiere saber qué experimentos se hicieron, quién los
dirigió, en qué laboratorio y a qué coste.


### Tarea 1.1: crea el esquema y las 3 tablas

Crea el esquema `dm_experimentos` y estas 3 tablas. Fíjate en las **FK internas**
del mart: `experimentos` referencia a `investigadores` y `laboratorios`.

**`laboratorios`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| lab_id | INTEGER | PRIMARY KEY |
| lab_name | VARCHAR | NOT NULL |
| country | VARCHAR | |

**`investigadores`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| researcher_id | INTEGER | PRIMARY KEY |
| researcher_name | VARCHAR | NOT NULL |
| department | VARCHAR | |

**`experimentos`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| experiment_id | INTEGER | PRIMARY KEY |
| experiment_name | VARCHAR | NOT NULL |
| researcher_id | INTEGER | FOREIGN KEY -> investigadores(researcher_id) |
| lab_id | INTEGER | FOREIGN KEY -> laboratorios(lab_id) |
| project_id | INTEGER | clave de negocio, sin FK |
| cost_eur | DECIMAL(10,2) | |
| status | VARCHAR | |


In [ ]:
# TODO: Crea el esquema dm_experimentos
# TODO: Crea las tablas laboratorios, investigadores y experimentos
#       (experimentos tiene FK a investigadores y laboratorios)

### El modelo de `dm_experimentos`

```mermaid
erDiagram
    LABORATORIOS ||--o{ EXPERIMENTOS : FK-lab
    INVESTIGADORES ||--o{ EXPERIMENTOS : FK-researcher
    LABORATORIOS {
        int lab_id PK
        string lab_name
        string country
    }
    INVESTIGADORES {
        int researcher_id PK
        string researcher_name
        string department
    }
    EXPERIMENTOS {
        int experiment_id PK
        int researcher_id FK
        int lab_id FK
        int project_id
        float cost_eur
        string status
    }
```

`project_id` aparece en `EXPERIMENTOS` pero **sin FK**: apunta a un proyecto de
otro mart (`dm_finance`). Ese enlace lo cruzará el Data Warehouse.


### Tarea 1.2: inserta los datos

**`laboratorios`**

| lab_id | lab_name | country |
|--------|----------|---------|
| 3 | Arklay Research Lab | USA |
| 7 | Tanuki BioPlant | Japan |
| 9 | Spencer R&D Annex | Switzerland |

**`investigadores`**

| researcher_id | researcher_name | department |
|---------------|-----------------|------------|
| 14 | Rebecca Chambers | Bioresearch |
| 22 | Marcus Aiken | Bioresearch |
| 30 | Ada Wong | Advanced Research |
| 31 | Leon Kennedy | Security |
| 40 | Claire Redfield | Bioresearch |

**`experimentos`**

| experiment_id | experiment_name | researcher_id | lab_id | project_id | cost_eur | status |
|---------------|-----------------|---------------|--------|------------|----------|--------|
| 1 | Alpha Test | 14 | 3 | 1 | 15000.00 | Completed |
| 2 | Beta Trial | 22 | 7 | 2 | 22000.00 | Completed |
| 3 | Gamma Study | 30 | 9 | 3 | 18500.00 | Failed |
| 4 | Delta Analysis | 31 | 3 | 4 | 9500.00 | Completed |
| 5 | Epsilon Probe | 40 | 7 | 5 | 31000.00 | In Progress |


In [ ]:
# TODO: Inserta los 3 laboratorios, los 5 investigadores y los 5 experimentos

---

## Parte 2: el data mart `dm_bioresearch`

El área de **Bioresearch** quiere saber qué muestras se usan, de qué tipo son y
qué riesgo tienen.


### Tarea 2.1: crea el esquema y las 3 tablas

Crea el esquema `dm_bioresearch` y estas 3 tablas.

**`donantes`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| donante_id | INTEGER | PRIMARY KEY |
| tipo | VARCHAR | NOT NULL |
| pais | VARCHAR | |

**`tipos_muestra`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| sample_type | VARCHAR | PRIMARY KEY |
| descripcion | VARCHAR | |
| temp_conservacion_c | INTEGER | |

**`muestras`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| sample_id | INTEGER | PRIMARY KEY |
| experiment_id | INTEGER | clave de negocio, sin FK |
| donante_id | INTEGER | FOREIGN KEY -> donantes(donante_id) |
| sample_type | VARCHAR | FOREIGN KEY -> tipos_muestra(sample_type) |
| risk_level | VARCHAR | |


In [ ]:
# TODO: Crea el esquema dm_bioresearch
# TODO: Crea las tablas donantes, tipos_muestra y muestras
#       (muestras tiene FK a donantes y tipos_muestra)

### El modelo de `dm_bioresearch`

```mermaid
erDiagram
    DONANTES ||--o{ MUESTRAS : FK-donante
    TIPOS_MUESTRA ||--o{ MUESTRAS : FK-tipo
    DONANTES {
        int donante_id PK
        string tipo
        string pais
    }
    TIPOS_MUESTRA {
        string sample_type PK
        string descripcion
        int temp_conservacion_c
    }
    MUESTRAS {
        int sample_id PK
        int experiment_id
        int donante_id FK
        string sample_type FK
        string risk_level
    }
```

`experiment_id` es otra **clave de negocio**: une cada muestra con su experimento
en `dm_experimentos`, pero **sin FK** entre esquemas.


### Tarea 2.2: inserta los datos

**`donantes`**

| donante_id | tipo | pais |
|------------|------|------|
| 1 | Humano | Spain |
| 2 | Humano | USA |
| 3 | Animal | Japan |
| 4 | Ambiente | Germany |

**`tipos_muestra`**

| sample_type | descripcion | temp_conservacion_c |
|-------------|-------------|---------------------|
| Blood | Sangre completa | -80 |
| Tissue | Tejido biológico | -80 |
| Saliva | Muestra de saliva | 4 |
| DNA | ADN purificado | -20 |

**`muestras`**

| sample_id | experiment_id | donante_id | sample_type | risk_level |
|-----------|---------------|------------|-------------|------------|
| 101 | 1 | 1 | Blood | High |
| 102 | 2 | 2 | Tissue | Medium |
| 103 | 3 | 3 | Saliva | Low |
| 104 | 4 | 1 | Blood | High |
| 105 | 5 | 4 | DNA | Medium |


In [ ]:
# TODO: Inserta los 4 donantes, los 4 tipos de muestra y las 5 muestras

---

## Parte 3: el data mart `dm_finance`

El área de **Finanzas** quiere controlar presupuestos y gastos por proyecto.


### Tarea 3.1: crea el esquema y las 3 tablas

Crea el esquema `dm_finance` y estas 3 tablas.

**`departamentos`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| department | VARCHAR | PRIMARY KEY |
| director | VARCHAR | |
| centro_coste | VARCHAR | |

**`proyectos`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| project_id | INTEGER | PRIMARY KEY |
| project_name | VARCHAR | NOT NULL |
| department | VARCHAR | FOREIGN KEY -> departamentos(department) |
| priority | VARCHAR | |

**`gastos`**

| Columna | Tipo | Restricción |
|---------|------|-------------|
| gasto_id | INTEGER | PRIMARY KEY |
| project_id | INTEGER | FOREIGN KEY -> proyectos(project_id) |
| year | INTEGER | |
| budget_eur | DECIMAL(12,2) | |
| spent_eur | DECIMAL(12,2) | |


In [ ]:
# TODO: Crea el esquema dm_finance
# TODO: Crea las tablas departamentos, proyectos y gastos
#       (proyectos tiene FK a departamentos y gastos a proyectos)

### El modelo de `dm_finance`

```mermaid
erDiagram
    DEPARTAMENTOS ||--o{ PROYECTOS : FK-department
    PROYECTOS ||--o{ GASTOS : FK-project
    DEPARTAMENTOS {
        string department PK
        string director
        string centro_coste
    }
    PROYECTOS {
        int project_id PK
        string project_name
        string department FK
        string priority
    }
    GASTOS {
        int gasto_id PK
        int project_id FK
        int year
        float budget_eur
        float spent_eur
    }
```

`gastos` tiene **dos filas por proyecto** (2025 y 2026): una por año fiscal. Cuidado
al cruzar datos para no duplicar experimentos.


### Tarea 3.2: inserta los datos

**`departamentos`**

| department | director | centro_coste |
|------------|----------|--------------|
| Bioresearch | Albert Wesker | CC-100 |
| Advanced Research | William Birkin | CC-200 |
| IT | Annette Birkin | CC-300 |
| Security | HUNK | CC-400 |

**`proyectos`**

| project_id | project_name | department | priority |
|------------|--------------|------------|----------|
| 1 | Alpha Protocol | Bioresearch | High |
| 2 | Beta Shield | Advanced Research | Critical |
| 3 | Gamma Guard | Bioresearch | Medium |
| 4 | Delta Stream | IT | Low |
| 5 | Epsilon Lab | Security | Medium |

**`gastos`**

| gasto_id | project_id | year | budget_eur | spent_eur |
|----------|------------|------|------------|-----------|
| 1 | 1 | 2025 | 90000.00 | 87000.00 |
| 2 | 1 | 2026 | 100000.00 | 45000.00 |
| 3 | 2 | 2025 | 140000.00 | 138000.00 |
| 4 | 2 | 2026 | 150000.00 | 120000.00 |
| 5 | 3 | 2025 | 75000.00 | 70000.00 |
| 6 | 3 | 2026 | 80000.00 | 35000.00 |
| 7 | 4 | 2025 | 45000.00 | 40000.00 |
| 8 | 4 | 2026 | 50000.00 | 12000.00 |
| 9 | 5 | 2025 | 55000.00 | 54000.00 |
| 10 | 5 | 2026 | 60000.00 | 55000.00 |


In [ ]:
# TODO: Inserta los 4 departamentos, los 5 proyectos y los 10 gastos

---

## Parte 4: el Data Warehouse como **vistas**

Los 3 marts son **independientes**: no hay ninguna FK entre ellos. Los enlaces entre
áreas (`experimentos.experiment_id` con `muestras.experiment_id`,
`experimentos.project_id` con `proyectos.project_id`) son **claves de negocio**:
columnas que comparten el mismo significado.

El **Data Warehouse** es la capa que integra esos marts. Lo construimos con
**vistas** (`CREATE VIEW`): una vista es una consulta guardada que **no copia
datos**; cada vez que la consultas se calcula sobre los marts. Si mañana cambia un
mart, la vista lo refleja sola.

```mermaid
flowchart TB
    E["dm_experimentos<br/>experimentos"]
    M["dm_bioresearch<br/>muestras"]
    G["dm_finance<br/>gastos y proyectos"]
    V1["dw.v_experimentos_full<br/>JOIN por experiment_id y project_id"]
    V2["dw.v_coste_por_laboratorio"]
    V3["dw.v_muestras_alto_riesgo"]
    E --> V1
    M --> V1
    G --> V1
    V1 --> V2
    V1 --> V3
```


### Tarea 4.1: `dw.v_experimentos_full`

Crea el esquema `dw` y la vista `v_experimentos_full`, que une los 3 marts con
`JOIN` sobre las claves de negocio. Usa el gasto del año **2026** para no duplicar
experimentos.

Debe devolver estas columnas:

| Columna | Viene de |
|---------|----------|
| experiment_id, experiment_name, status, cost_eur | dm_experimentos.experimentos |
| researcher_name, researcher_department | dm_experimentos.investigadores |
| lab_name, lab_country | dm_experimentos.laboratorios |
| sample_type, risk_level | dm_bioresearch.muestras |
| project_name, project_priority | dm_finance.proyectos |
| budget_eur, spent_eur | dm_finance.gastos (year = 2026) |


In [ ]:
# TODO: Crea el esquema dw y la vista v_experimentos_full
#       (JOIN sobre experiment_id y project_id, gastos del año 2026)

### Tarea 4.2: `dw.v_coste_por_laboratorio`

Una vista puede construirse **sobre otra vista** (no se copian datos). Crea
`v_coste_por_laboratorio` sobre `v_experimentos_full` con las columnas `lab_name`,
`n_experimentos` (COUNT), `coste_total` (SUM de cost_eur) y `coste_medio` (AVG),
agrupadas por `lab_name`.


In [ ]:
# TODO: Crea la vista v_coste_por_laboratorio sobre v_experimentos_full

### Tarea 4.3: `dw.v_muestras_alto_riesgo`

Crea `v_muestras_alto_riesgo` sobre `v_experimentos_full`, con las columnas
`experiment_id`, `experiment_name`, `sample_type`, `risk_level`, `researcher_name`
y `cost_eur`, filtrando `risk_level = 'High'`.


In [ ]:
# TODO: Crea la vista v_muestras_alto_riesgo sobre v_experimentos_full

---

## Verificación: consulta tu Data Warehouse


In [ ]:
# Verificacion: consulta tu Data Warehouse
print('=== dw.v_experimentos_full ===')
display(con.execute('''
    SELECT experiment_id, experiment_name, lab_name, researcher_name,
           sample_type, risk_level, project_name, budget_eur, spent_eur, cost_eur
    FROM dw.v_experimentos_full
    ORDER BY experiment_id
''').fetchdf())

print('=== dw.v_coste_por_laboratorio ===')
display(con.execute('SELECT * FROM dw.v_coste_por_laboratorio ORDER BY lab_name').fetchdf())

print('=== dw.v_muestras_alto_riesgo ===')
display(con.execute('SELECT * FROM dw.v_muestras_alto_riesgo ORDER BY experiment_id').fetchdf())

---

## Guarda tu trabajo

Copia todas tus sentencias (`CREATE SCHEMA`, `CREATE TABLE`, `INSERT INTO` y
`CREATE VIEW`) en `scripts/data_marts.py`. El fichero debe:

1. Conectar a DuckDB en memoria
2. Crear los 3 esquemas con sus tablas y datos
3. Crear el esquema `dw` con sus 3 vistas

## Verificación

```bash
pytest tests/ -v
```


---

## Resumen

Has creado 3 Data Marts en esquemas independientes y un Data Warehouse que los
integra con vistas.

```mermaid
erDiagram
    LABORATORIOS ||--o{ EXPERIMENTOS : FK-lab
    INVESTIGADORES ||--o{ EXPERIMENTOS : FK-researcher
    DONANTES ||--o{ MUESTRAS : FK-donante
    TIPOS_MUESTRA ||--o{ MUESTRAS : FK-tipo
    DEPARTAMENTOS ||--o{ PROYECTOS : FK-department
    PROYECTOS ||--o{ GASTOS : FK-project
    EXPERIMENTOS ||..o{ MUESTRAS : negocio-experiment_id
    PROYECTOS ||..o{ EXPERIMENTOS : negocio-project_id
    LABORATORIOS {
        int lab_id PK
        string lab_name
    }
    INVESTIGADORES {
        int researcher_id PK
        string researcher_name
    }
    EXPERIMENTOS {
        int experiment_id PK
        int researcher_id FK
        int lab_id FK
        int project_id
        float cost_eur
    }
    DONANTES {
        int donante_id PK
        string tipo
    }
    TIPOS_MUESTRA {
        string sample_type PK
        string descripcion
    }
    MUESTRAS {
        int sample_id PK
        int experiment_id
        int donante_id FK
        string sample_type FK
        string risk_level
    }
    DEPARTAMENTOS {
        string department PK
        string director
    }
    PROYECTOS {
        int project_id PK
        string project_name
        string department FK
    }
    GASTOS {
        int gasto_id PK
        int project_id FK
        int year
        float budget_eur
    }
```

En el diagrama, las líneas **continuas** son FK reales dentro de cada mart. Las
**discontinuas** son claves de negocio: columnas que comparten dos marts sin FK, y
que el DW cruza con `JOIN`.


### Siguiente: Actividad 04: Esquema Estrella y Arquitectura
